In [ ]:
import model_inference as mi
from importlib import reload
import pandas as pd
from data_extraction import main as extract_data
from feature_derivation import main as derive_features

#data_folder = extract_data()

data_folder = "Data_16_06_2026"

# X_train, y_train, X_inference = derive_features(
#     data_folder=data_folder
# )

# X_train.to_parquet(f'{data_folder}/preprocessed_data/X_train.parquet')
# y_train.to_frame().to_parquet(f'{data_folder}/preprocessed_data/y_train.parquet')
# X_inference.to_parquet(f'{data_folder}/preprocessed_data/X_inference.parquet')

X_train = pd.read_parquet(f'{data_folder}/preprocessed_data/X_train.parquet')
y_train = pd.read_parquet(f'{data_folder}/preprocessed_data/y_train.parquet').close_log_return
X_inference = pd.read_parquet(f'{data_folder}/preprocessed_data/X_inference.parquet')

In [ ]:
params = {"n_estimators": 1497, "learning_rate": 0.234205, "max_depth": 12, "subsample": 0.760059, "colsample_bytree": 0.495811, "colsample_bynode": 0.860969, "min_child_weight": 5, "gamma": 0.630154, "reg_alpha": 0.038199, "reg_lambda": 2e-06, "num_parallel_tree": 20}

preds, metrics = mi.back_test(
    X_train,
    y_train,
    whole_back_test=True,
    **params
)

100%|██████████| 16/16 [2:29:16<00:00, 559.77s/it]  


In [60]:
import numpy as np
preds.groupby(
    pd.cut(
        preds.y_pred,
        bins=[-1] + list(np.arange(-0.5, 1.1, 0.1)) + [1.5, 2, 25]
    )
).y_true.describe()

,count,mean,std,min,25%,50%,75%,max
y_pred,,,,,,,,
"(-1.0, -0.5]",20634.0,-0.290840,1.367633,-0.999000,-0.848885,-0.619048,-0.204313,46.153843
"(-0.5, -0.4]",16277.0,-0.126509,1.271848,-0.999000,-0.667939,-0.377014,0.023256,62.949574
"(-0.4, -0.3]",21916.0,-0.028499,1.478327,-0.999000,-0.555556,-0.250000,0.140622,105.711136
"(-0.3, -0.2]",28762.0,0.050506,1.187677,-0.999000,-0.417703,-0.109869,0.218712,89.000008
"(-0.2, -0.1]",40535.0,0.101705,0.968189,-0.998809,-0.250789,0.002043,0.264917,106.848862
"(-0.1, -1.11e-16]",59976.0,0.130830,0.649389,-0.997494,-0.143844,0.054822,0.286194,29.058331
"(-1.11e-16, 0.1]",72093.0,0.160304,0.571042,-0.994420,-0.083214,0.092277,0.309089,42.925930
"(0.1, 0.2]",35787.0,0.251945,0.679378,-0.989960,-0.008000,0.156165,0.389664,53.775002
"(0.2, 0.3]",7151.0,0.508043,1.558715,-0.962981,0.070838,0.308870,0.637391,104.821899


In [46]:
preds[preds.marketcap >= preds.marketcap_quantile].groupby('calendardate').tail(20).groupby('calendardate').y_true.mean()

calendardate
2010-03-31    0.928787
2010-06-30    1.056975
2010-09-30    0.315050
2010-12-31    0.143283
2011-03-31    0.220577
                ...   
2024-03-31    1.582990
2024-06-30    2.611912
2024-09-30    4.572766
2024-12-31    1.794924
2025-03-31    3.332789
Name: y_true, Length: 61, dtype: float32

In [ ]:
2**10 

1024

In [53]:
preds[
    (preds.close >= 0.25*preds.close_max)
    & (preds.marketcap >= preds.marketcap_quantile)
].groupby('calendardate').tail(20).groupby('calendardate').y_true.mean().describe()

count    61.000000
mean      0.962808
std       0.896065
min       0.065201
25%       0.312684
50%       0.723428
75%       1.177342
max       4.897539
Name: y_true, dtype: float64

In [58]:
preds[
    (preds.close >= 0.25*preds.close_max)
    & (preds.marketcap >= preds.marketcap_quantile)
].groupby('calendardate').tail(10).groupby('calendardate').y_true.mean().describe()

count    61.000000
mean      1.027672
std       1.130344
min      -0.106938
25%       0.361362
50%       0.752531
75%       1.152175
max       6.597761
Name: y_true, dtype: float64

In [3]:
params = {"n_estimators": 1497, "learning_rate": 0.234205, "max_depth": 12, "subsample": 0.760059, "colsample_bytree": 0.495811, "colsample_bynode": 0.860969, "min_child_weight": 5, "gamma": 0.630154, "reg_alpha": 0.038199, "reg_lambda": 2e-06, "num_parallel_tree": 20}

inference_predictions = mi.fit_and_predict(
    X_train,
    y_train,
    X_inference,
    **params
)

In [2]:
reload(mi)
inference_predictions = pd.read_csv('Data_16_06_2026/results/inference_predictions.csv')
inference_performance_to_date = mi.obtain_inference_performance_to_date(
    inference_predictions,
    marketcap_quantile=0.25,
    data_folder=data_folder
)

In [6]:
inference_predictions.to_parquet(f'{data_folder}/results/inference_predictions.parquet')

In [10]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
    & (inference_performance_to_date.calendardate == '2026-03-31')
    #& (inference_performance_to_date.ticker != 'SHAZ')
].sort_values('pct_change').groupby('calendardate').tail(20).pct_change_at_max_date.describe()#.pct_change_at_max_date.describe()

count    20.000000
mean      0.593059
std       0.738947
min      -0.151533
25%       0.061880
50%       0.377206
75%       0.641365
max       2.364423
Name: pct_change_at_max_date, dtype: float64

In [24]:
inference_performance_to_date[    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)].sort_values('pct_change').groupby('calendardate').tail(10).sort_values('calendardate')#.groupby('calendardate').pct_change_at_max_date.describe().T

,ticker,calendardate,pct_change,pct_change_at_max_date,close,close_max,marketcap,marketcap_quantile
9763,HUHU,2025-06-30,0.658131,0.652263,4.860000,5.960000,1.112663e+08,85688932.0
20959,ZBIO,2025-06-30,1.121637,0.930857,9.690000,25.680000,4.074649e+08,85688932.0
17528,SNDK,2025-06-30,1.005993,45.479824,45.349998,56.419998,5.785815e+09,85688932.0
12179,MAZE,2025-06-30,0.896648,0.977995,12.270000,15.950000,3.968023e+08,85688932.0
17214,SION,2025-06-30,1.139530,1.074928,17.350000,25.000000,5.692047e+08,85688932.0
11636,LION,2025-06-30,0.805424,1.473322,5.810000,8.150000,2.065744e+09,85688932.0
15923,RAPP,2025-06-30,0.755736,2.364996,11.370000,29.230000,3.886990e+08,85688932.0
560,DVS,2025-06-30,0.743145,-0.189349,3.380000,3.670000,2.027746e+08,85688932.0
16983,SEPN,2025-06-30,0.690438,2.408704,10.570000,27.090000,4.509304e+08,85688932.0
16249,RHLD,2025-06-30,0.649789,3.132413,31.870001,50.490002,2.397511e+08,85688932.0


In [16]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
    & (inference_performance_to_date.calendardate == '2026-03-31')
].sort_values('pct_change').groupby('calendardate').tail(10).pct_change_at_max_date.describe()

count    10.000000
mean      0.889677
std       0.916958
min      -0.151533
25%       0.376150
50%       0.495635
75%       1.714717
max       2.364423
Name: pct_change_at_max_date, dtype: float64

In [16]:
selected_stocks = [
    "NTSK",
    "INV",
    "MNTN",
    "CHA",
    "AGBK",
    "KLAR",
    "SSII",
    "BETA",
    "WOLF",
    "FLY",
    "EQPT",
    "WYFI",
    "OMDA",
    "TLX",
    "IBTA",
    "FIGR",
    "BBNX",
    "MANE",
    "LMRI",
    "PICS",
    "SAIL",
    "ANTA",
    "XZO",
    "ETOR",
    "PTRN",
    "AERO",
    "HTFL",
    "GLXY",
    "WLTH",
    "BLLN"
]

len(selected_stocks)

30

In [11]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
    #& (inference_performance_to_date.pct_change_at_max_date < 5)
].sort_values('pct_change').groupby('calendardate').tail(10).groupby('calendardate').pct_change_at_max_date.describe().T

calendardate,2025-06-30,2025-09-30,2025-12-31,2026-03-31
count,10.000000,10.000000,10.000000,10.000000
mean,8.749516,2.773352,0.882363,0.889677
std,13.473592,5.464709,0.468135,0.916958
min,0.930857,-0.115430,0.217290,-0.151533
25%,1.410271,0.322537,0.578400,0.376150
50%,4.342661,0.729436,0.880712,0.495635
75%,9.259249,1.754483,1.032720,1.714717
max,45.479824,17.786631,1.819644,2.364423


In [ ]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].groupby('calendardate').pct_change_at_max_date.describe().T

calendardate,2025-06-30,2025-09-30,2025-12-31,2026-03-31
count,2814.000000,2908.000000,2888.000000,2887.000000
mean,0.274981,0.156999,0.141795,0.122511
std,1.169675,0.648894,0.759396,0.313870
min,-0.999740,-0.992406,-0.983939,-0.808616
25%,-0.112840,-0.129635,-0.096226,-0.023179
50%,0.122893,0.066371,0.067051,0.061481
75%,0.405417,0.305204,0.239513,0.193922
max,45.479824,17.786631,32.238462,2.628977


In [17]:
inference_performance_to_date[
    inference_performance_to_date.ticker.isin(selected_stocks)
].sort_values('pct_change').groupby('calendardate').pct_change_at_max_date.describe().T

calendardate,2025-06-30,2025-09-30,2025-12-31,2026-03-31
count,9.000000,15.000000,23.000000,30.000000
mean,-0.269899,-0.144993,0.112739,0.284758
std,0.254457,0.232358,0.554429,0.477258
min,-0.517777,-0.522372,-0.528060,-0.296000
25%,-0.406347,-0.316364,-0.293867,0.013965
50%,-0.402377,-0.036831,0.098226,0.152547
75%,-0.119399,0.017431,0.427874,0.371248
max,0.252083,0.157271,1.819644,2.007966


In [12]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(5).groupby('calendardate').pct_change_at_max_date.mean()

calendardate
2025-06-30    12.957206
2025-09-30     5.099332
2025-12-31     7.227557
2026-03-31     1.069577
Name: pct_change_at_max_date, dtype: float64

In [ ]:
inference_performance_to_date[
    (inference_performance_to_date.close >= 0.25*inference_performance_to_date.close_max)
    & (inference_performance_to_date.marketcap >= inference_performance_to_date.marketcap_quantile)
].sort_values('pct_change').groupby('calendardate').tail(20).groupby('calendardate').pct_change_at_max_date.mean()b

calendardate
2025-03-31    2.134912
2025-06-30    2.092430
2025-09-30    0.568499
2025-12-31    0.217706
Name: pct_change_at_max_date, dtype: float64